# 🔧 Notebook 2 – Data Preprocessing

**Depends on:** `01_setup.ipynb` (must be run first in the same session)

**What this notebook does (in order):**

| Step | Description | Output |
|------|-------------|--------|
| 2.1 | Load shared config | CONFIG dict |
| 2.2 | Copy mesh files | `data/XX/mesh.ply` |
| 2.3 | Generate 9 keypoints via FPS | `data/XX/Outside9.npy` |
| 2.4 | Extract poses from `gt.yml` | `data/XX/pose/poseXXXXXX.npy` |
| 2.5 | Rename files to 6-digit format | `rgb/000000.png`, `depth/000000.dpt` |
| 2.6 | Convert depth `.png` → `.dpt` | binary depth files |
| 2.7 | Generate 9 radius maps per frame | `data/XX/Out_ptN_dm/XXXXXX.npy` |
| 2.8 | Split data 70 / 20 / 10 % | `Split/train.txt`, `val.txt`, `test.txt` |
| 2.9 | Normalise units + sanity check | All files verified in metres |

**Next step → `03_yolo_train.ipynb`**

## Cell 2.1 – Load shared config from `01_setup.ipynb`

Every notebook starts by reading `/content/config.json` that was written by `01_setup.ipynb`. This avoids copy-pasting paths across notebooks.

In [ ]:
import os, json, sys, random
import numpy as np

with open('/content/config.json') as f:
    CONFIG = json.load(f)

DATA_DIR      = CONFIG['DATA_DIR']
MODELS_PLY    = CONFIG['MODELS_DIR_PLY']
ALL_CLASSES   = CONFIG['ALL_CLASSES']
REPO_DIR      = CONFIG['REPO_DIR']
CAMERA_K      = np.array(CONFIG['CAMERA_K'], dtype=np.float64)

# Add repo to Python path so src/ modules are importable
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)

random.seed(42)
np.random.seed(42)

print('✅ Config loaded.')
print(f'   DATA_DIR  = {DATA_DIR}')
print(f'   Classes   = {ALL_CLASSES}')

## Cell 2.2 – Copy 3D model files: `models/obj_XX.ply` → `data/XX/mesh.ply`

The evaluation and visualization steps expect the mesh at `data/XX/mesh.ply`. The source files live in `Linemod_preprocessed/models/` as `obj_01.ply`, `obj_02.ply` …

`shutil.copy2` preserves file timestamps.

In [ ]:
import shutil

for filename in sorted(os.listdir(MODELS_PLY)):
    if not (filename.startswith('obj_') and filename.endswith('.ply')):
        continue
    model_id = filename.split('_')[1].split('.')[0]   # 'obj_05.ply' → '05'
    dst_dir  = os.path.join(DATA_DIR, model_id)
    if not os.path.isdir(dst_dir):
        continue
    dst_file = os.path.join(dst_dir, 'mesh.ply')
    shutil.copy2(os.path.join(MODELS_PLY, filename), dst_file)
    print(f'   {filename}  →  data/{model_id}/mesh.ply')

print('\n✅ All mesh files copied.')

## Cell 2.3 – Generate 9 keypoints via Farthest-Point Sampling (FPS)

We select 9 keypoints that are maximally spread across each 3D object model. These points are used to build per-frame radius maps (distances from every foreground pixel to each of the 9 keypoints in 3D).

**FPS algorithm:** Start with a random point, then repeatedly pick the point that is farthest from the current selected set. This guarantees good spatial coverage.

**Result:** `Outside9.npy` — shape `(9, 3)`, values in mm (same units as the PLY).

In [ ]:
import open3d as o3d

def farthest_point_sampling(points: np.ndarray, k: int, seed: int = 42) -> np.ndarray:
    """Select k points maximally spread over the point cloud."""
    np.random.seed(seed)
    N          = len(points)
    selected   = np.zeros(k, dtype=np.int32)
    min_dists  = np.full(N, np.inf)
    farthest   = np.random.randint(0, N)
    for i in range(k):
        selected[i] = farthest
        d           = np.sum((points - points[farthest]) ** 2, axis=1)
        min_dists   = np.minimum(min_dists, d)
        farthest    = int(np.argmax(min_dists))
    return points[selected]

for cls in ALL_CLASSES:
    out_path  = os.path.join(DATA_DIR, cls, 'Outside9.npy')
    mesh_path = os.path.join(DATA_DIR, cls, 'mesh.ply')
    if os.path.exists(out_path):
        print(f'   ⏩ Class {cls}: Outside9.npy already exists')
        continue
    if not os.path.isfile(mesh_path):
        print(f'   ⚠️  Class {cls}: mesh.ply not found')
        continue
    pts  = np.asarray(o3d.io.read_point_cloud(mesh_path).points)
    kpts = farthest_point_sampling(pts, k=9)
    np.save(out_path, kpts)
    print(f'   ✅ Class {cls}: Outside9.npy  shape={kpts.shape}')

print('\n✅ Keypoints ready.')

## Cell 2.4 – Extract ground-truth poses from `gt.yml`

`gt.yml` structure (per image ID):
- `cam_R_m2c`: `[9 floats]` — rotation matrix (object → camera)
- `cam_t_m2c`: `[3 floats]` — translation in **mm**

We build a `(3, 4)` matrix `[R | t]` and save as `pose/poseXXXXXX.npy`. Translation is divided by 1000 to convert **mm → metres**.

When a frame has multiple object instances (rare), we use index `[0]`.

In [ ]:
import yaml
from tqdm import tqdm

for cls in ALL_CLASSES:
    cls_dir  = os.path.join(DATA_DIR, cls)
    gt_file  = os.path.join(cls_dir, 'gt.yml')
    pose_dir = os.path.join(cls_dir, 'pose')
    if not os.path.isfile(gt_file):
        print(f'   ⚠️  Class {cls}: gt.yml not found')
        continue
    os.makedirs(pose_dir, exist_ok=True)
    with open(gt_file) as f:
        gt = yaml.safe_load(f)
    for img_id, entries in tqdm(gt.items(), desc=f'Class {cls}', leave=False):
        e     = entries[0]
        R_mat = np.array(e['cam_R_m2c'], dtype=np.float32).reshape(3, 3)
        t_vec = np.array(e['cam_t_m2c'], dtype=np.float32).reshape(3, 1) / 1000.0  # mm→m
        RT    = np.hstack([R_mat, t_vec])         # shape (3, 4)
        np.save(os.path.join(pose_dir, f'pose{int(img_id):06d}.npy'), RT)
    print(f'   ✅ Class {cls}: {len(gt)} pose files saved')

print('\n✅ Pose extraction complete.')

## Cell 2.5 – Rename files from 4-digit to 6-digit zero-padded names

LineMOD stores images as `0000.png` (4 digits). The dataset loader and radius-map generator expect `000000.png` (6 digits).

This cell renames files inside `rgb/`, `mask/`, and `depth/` folders. Only files whose stem is purely numeric are renamed (safe guard).

In [ ]:
for cls in ALL_CLASSES:
    for folder in ['rgb', 'mask', 'depth']:
        folder_path = os.path.join(DATA_DIR, cls, folder)
        if not os.path.isdir(folder_path):
            continue
        for fname in sorted(os.listdir(folder_path)):
            stem, ext = os.path.splitext(fname)
            if not stem.isdigit():
                continue
            new_name = f'{int(stem):06d}{ext}'
            if fname != new_name:
                os.rename(os.path.join(folder_path, fname),
                          os.path.join(folder_path, new_name))
    print(f'   ✅ Class {cls}: renamed to 6-digit format')

print('\n✅ All files renamed.')

## Cell 2.6 – Convert depth images: `.png` (16-bit) → `.dpt` (binary)

LineMOD depth images are 16-bit PNG files storing depth in millimetres. We convert them to a compact binary `.dpt` format:
```
[uint32 H][uint32 W][uint16 × H×W   depth values in mm]
```

**Benefits of `.dpt`:**
- Faster to read (no PNG decompression overhead)
- Smaller file size

The original `.png` is deleted after conversion to save ~50% disk space.

In [ ]:
import struct, cv2

def png_to_dpt(png_path: str, dpt_path: str):
    depth = cv2.imread(png_path, cv2.IMREAD_UNCHANGED).astype(np.uint16)
    h, w  = depth.shape
    with open(dpt_path, 'wb') as f:
        f.write(struct.pack('II', h, w))
        f.write(depth.tobytes())
    os.remove(png_path)   # delete the .png to free disk space

for cls in ALL_CLASSES:
    depth_dir = os.path.join(DATA_DIR, cls, 'depth')
    if not os.path.isdir(depth_dir):
        continue
    pngs = sorted(f for f in os.listdir(depth_dir) if f.endswith('.png'))
    for fname in tqdm(pngs, desc=f'Class {cls}', leave=False):
        png = os.path.join(depth_dir, fname)
        dpt = png.replace('.png', '.dpt')
        png_to_dpt(png, dpt)
    print(f'   ✅ Class {cls}: {len(pngs)} depth files → .dpt')

print('\n✅ All depth images converted.')

## Cell 2.7 – Generate 9 radius maps per frame (Numba JIT-accelerated)

A radius map for keypoint `k` at frame `i` stores, for every foreground pixel, the 3D Euclidean distance (in metres) to keypoint `k` projected into camera space using the ground-truth pose.

**Per frame and per keypoint:**
1. Read depth `.dpt` → back-project pixels to 3D using intrinsics K
2. Apply mask (set background pixels to 0)
3. Project keypoint with GT pose: `kp_cam = R @ kp_obj + t`
4. Distance = `‖pixel_3D − kp_cam‖₂`
5. Save as `float32` `.npy` (metres)

Numba `@jit(parallel=True)` processes all pixels in parallel across CPU cores. Without Numba this loop would take ~10 hours; with it: **~10–20 minutes**.

In [ ]:
from numba import jit, prange
from PIL import Image as PILImage

@jit(nopython=True, parallel=True, fastmath=True, cache=True)
def compute_radius_map(depth, kp_cam, fx, fy, cx, cy):
    """
    Compute per-pixel 3D distance to a single camera-space keypoint.
    depth  : (H, W) float64, depth in mm
    kp_cam : (3,)   float64, keypoint in camera space (mm)
    returns: (H, W) float64, distances in mm (0 for background)
    """
    H, W   = depth.shape
    result = np.zeros((H, W), dtype=np.float64)
    for v in prange(H):
        for u in range(W):
            z = depth[v, u]
            if z == 0.0:
                continue
            x  = (u - cx) * z / fx
            y  = (v - cy) * z / fy
            dx = x - kp_cam[0]
            dy = y - kp_cam[1]
            dz = z - kp_cam[2]
            result[v, u] = (dx*dx + dy*dy + dz*dz) ** 0.5
    return result

def read_dpt(path):
    with open(path, 'rb') as f:
        h, w = struct.unpack('II', f.read(8))
        data = np.frombuffer(f.read(h * w * 2), dtype=np.uint16)
    return data.reshape(h, w).astype(np.float64)

fx, fy = CAMERA_K[0, 0], CAMERA_K[1, 1]
cx, cy = CAMERA_K[0, 2], CAMERA_K[1, 2]

for cls in ALL_CLASSES:
    cls_dir   = os.path.join(DATA_DIR, cls)
    depth_dir = os.path.join(cls_dir, 'depth')
    mask_dir  = os.path.join(cls_dir, 'mask')
    pose_dir  = os.path.join(cls_dir, 'pose')
    kp_path   = os.path.join(cls_dir, 'Outside9.npy')

    if not os.path.isfile(kp_path):
        print(f'   ⚠️  Class {cls}: Outside9.npy missing')
        continue

    keypoints = np.load(kp_path)    # (9, 3) mm  (same unit as PLY mesh)
    dpt_files = sorted(f for f in os.listdir(depth_dir) if f.endswith('.dpt'))
    print(f'\n📂 Class {cls}: {len(keypoints)} keypoints × {len(dpt_files)} frames')

    for pt_idx, kp in enumerate(keypoints, start=1):
        out_dir = os.path.join(cls_dir, f'Out_pt{pt_idx}_dm')
        os.makedirs(out_dir, exist_ok=True)

        for fname in tqdm(dpt_files, desc=f'  kp{pt_idx}', leave=False):
            base     = os.path.splitext(fname)[0]
            out_path = os.path.join(out_dir, f'{base}.npy')
            if os.path.exists(out_path):
                continue

            pose_p = os.path.join(pose_dir, f'pose{base}.npy')
            mask_p = os.path.join(mask_dir, f'{base}.png')
            if not os.path.exists(pose_p) or not os.path.exists(mask_p):
                continue

            depth = read_dpt(os.path.join(depth_dir, fname))
            mask  = np.asarray(PILImage.open(mask_p).convert('L'))
            depth[mask == 0] = 0.0

            RT     = np.load(pose_p).astype(np.float64)     # (3,4) R | t(m)
            t_mm   = RT[:, 3] * 1000.0                      # back to mm for consistency
            RT_mm  = RT.copy(); RT_mm[:, 3] = t_mm
            kp_cam = RT_mm[:, :3] @ kp + RT_mm[:, 3]       # (3,) mm

            rmap = compute_radius_map(depth, kp_cam, fx, fy, cx, cy)
            np.save(out_path, (rmap / 1000.0).astype(np.float32))

print('\n✅ All radius maps generated.')

## Cell 2.8 – Split data: 70% train / 20% val / 10% test

For each class we collect all frame IDs that have:
- An RGB image (`rgb/XXXXXX.png`)
- A pose file (`pose/poseXXXXXX.npy`)

We shuffle with `seed=42` for reproducibility and write three text files:
- `Split/train.txt` — 70% of frames
- `Split/val.txt` — 20% of frames
- `Split/test.txt` — 10% of frames

Format: one 6-digit frame ID per line (e.g. `"000123"`).

In [ ]:
import glob

random.seed(42)

for cls in ALL_CLASSES:
    cls_dir   = os.path.join(DATA_DIR, cls)
    rgb_dir   = os.path.join(cls_dir, 'rgb')
    pose_dir  = os.path.join(cls_dir, 'pose')
    split_dir = os.path.join(cls_dir, 'Split')
    os.makedirs(split_dir, exist_ok=True)

    pose_ids = {
        os.path.basename(p).replace('pose', '').replace('.npy', '')
        for p in glob.glob(os.path.join(pose_dir, 'pose*.npy'))
    }
    valid_ids = sorted([
        os.path.splitext(os.path.basename(p))[0]
        for p in glob.glob(os.path.join(rgb_dir, '*.png'))
        if os.path.splitext(os.path.basename(p))[0] in pose_ids
    ])

    random.shuffle(valid_ids)
    n     = len(valid_ids)
    cut1  = int(0.70 * n)
    cut2  = int(0.90 * n)

    splits = {'train': valid_ids[:cut1],
              'val'  : valid_ids[cut1:cut2],
              'test' : valid_ids[cut2:]}

    for name, ids in splits.items():
        with open(os.path.join(split_dir, f'{name}.txt'), 'w') as f:
            f.write('\n'.join(ids))

    print(f'   ✅ Class {cls}: train={len(splits["train"])},'
          f' val={len(splits["val"])}, test={len(splits["test"])}')

print('\n✅ Data splits written.')

## Cell 2.9 – Normalisation & sanity check

This step scans all preprocessed files and fixes unit inconsistencies:

1. **Radius maps** (`Out_pt*_dm/*.npy`) — if values `> 20` → assume mm, divide by 1000 to convert to metres. Files already in metres are left untouched.
2. **Pose files** (`pose/poseXXXXXX.npy`) — if `‖t‖ > 20` → assume mm, divide by 1000. Already-normalised files are left untouched.
3. **RGB images** — verifies each file is loadable.
4. **Depth files** — verifies each `.dpt` file is readable.
5. **Mask files** — verifies each mask is loadable.

> ℹ️ Running this cell is **idempotent**: re-running it on already-normalised data makes no changes.

In [ ]:
RADIUS_UNIT_SCALE = 1000.0
OUT_PTS           = [f'Out_pt{i}_dm' for i in range(1, 10)]

def _sep(char='─'):
    print(char * 70)

for cls in ALL_CLASSES:
    cls_dir = os.path.join(DATA_DIR, cls)
    if not os.path.isdir(cls_dir):
        continue

    _sep()
    print(f'📂  Class {cls}'.center(70, '─'))

    # 1. Radius maps
    print('🌐  Checking radius maps …')
    for out_dir in OUT_PTS:
        dir_path = os.path.join(cls_dir, out_dir)
        if not os.path.isdir(dir_path):
            continue
        files = [f for f in os.listdir(dir_path) if f.endswith('.npy')]
        changed = 0
        for fn in files:
            path = os.path.join(dir_path, fn)
            try:
                arr = np.load(path)
                if arr.size == 0:
                    continue
                if arr.max() > 20.0:
                    arr = arr / RADIUS_UNIT_SCALE
                    np.save(path, arr.astype(np.float32))
                    changed += 1
            except Exception as e:
                print(f'   ⚠️  [{out_dir}/{fn}] {e}')
        if changed:
            print(f'   ✅ {out_dir}: {changed} file(s) scaled mm→m')

    # 2. Pose translation
    print('🧭  Checking pose translations …')
    pose_dir = os.path.join(cls_dir, 'pose')
    if os.path.isdir(pose_dir):
        changed = 0
        for fn in [f for f in os.listdir(pose_dir) if f.endswith('.npy')]:
            path = os.path.join(pose_dir, fn)
            try:
                pose = np.load(path).astype(np.float32)
                t = pose[:3, 3]
                if np.linalg.norm(t) > 20:
                    pose[:3, 3] = t / 1000.0
                    np.save(path, pose)
                    changed += 1
            except Exception as e:
                print(f'   ⚠️  [{fn}] {e}')
        if changed:
            print(f'   ✅ {changed} pose file(s) translation scaled mm→m')
        else:
            print('   ✅ All pose translations already in metres')

    # 3–5. RGB / Depth / Mask sanity checks
    split_dir = os.path.join(cls_dir, 'Split')
    for split_file in ('train.txt', 'val.txt', 'test.txt'):
        split_path = os.path.join(split_dir, split_file)
        if not os.path.isfile(split_path):
            continue
        ids = [l.strip() for l in open(split_path) if l.strip()]
        if not ids:
            continue

        rgb_ok = depth_ok = mask_ok = 0
        for sid in ids:
            try:
                _ = PILImage.open(os.path.join(cls_dir, 'rgb', f'{sid}.png')).convert('RGB')
                rgb_ok += 1
            except Exception:
                pass
            try:
                _ = read_dpt(os.path.join(cls_dir, 'depth', f'{sid}.dpt'))
                depth_ok += 1
            except Exception:
                pass
            try:
                _ = PILImage.open(os.path.join(cls_dir, 'mask', f'{sid}.png')).convert('L')
                mask_ok += 1
            except Exception:
                pass

        n = len(ids)
        print(f'   📋 {split_file:<10} │ RGB {rgb_ok}/{n}  '
              f'Depth {depth_ok}/{n}  Mask {mask_ok}/{n}')

    _sep('=')
    print(f'✅ Class {cls} normalisation complete'.center(70))
    _sep('=')

print('\n✅ All classes normalised and verified.')

---
## ✅ Preprocessing Complete

For each class folder:
- ✅ `mesh.ply` copied
- ✅ `Outside9.npy` generated
- ✅ `pose/poseXXXXXX.npy` files created
- ✅ All files renamed to 6-digit format
- ✅ `depth/*.dpt` binary files created
- ✅ `Out_pt1_dm/` … `Out_pt9_dm/` radius maps generated
- ✅ `Split/train.txt`, `val.txt`, `test.txt` written
- ✅ All units normalised to metres + RGB/Depth/Mask verified

**Next step → Open and run `03_yolo_train.ipynb`**